# Datasets

Summary of some of the datasets used in this study.

This notebook provides demonstrations on newly downloaded datasets; preprocessed versions of these datasets, containing splits, Butina clusters, etc. are in the [data directory of this repo](../data)

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
from rdkit import Chem, DataStructs
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

import topological_pretraining as tp

In [ ]:
temp_dir = "./temp"
os.makedirs(temp_dir, exist_ok=True)
data_dir = "../data"

## Example with Biogen

We use primarily use pandas DataFrames with a few additionaly methods and properties to handle our data.

Here, we can use [Biogen](https://github.com/molecularinformatics/Computational-ADME/blob/main/ADME_public_set_3521.csv) as an example.

### Loading dataset

In [ ]:
standardizer = tp.data.mol.Standardizer(n_jobs=-1)
biogen = tp.data.load_dataset("Biogen", verbose=True, standardizer=standardizer, root=temp_dir)
biogen.head()

In [ ]:
f"Number of molecules in Biogen dataset: {len(biogen.rdkit_mols)}"

In [ ]:
f"Number of molecules that failed rdkit standardization: \
{len(biogen[~biogen.rdkit_pass])}"

### Inspecting molecules and tasks

We can assess pre-computed standardized rdkit molecules using `rdkit_mols`:

In [ ]:
Chem.Draw.MolsToGridImage(biogen.rdkit_mols[:6], molsPerRow=3)

Biogen has six tasks:
- Human plasma protein binding (human_ppb)
- Rat plasma protein binding (rat_ppb)
- Human intrinsic hepatic clearance (human_clint)
- Rat Human intrinsic hepatic clearance (rat_clint)
- Aqueous solubility (solu)
- MDCK-MDR1 permeability as an efflux ratio (efflux)

In [ ]:
tasks = [
    "human_ppb",
    "rat_ppb",
    "human_clint",
    "rat_clint",
    "solu",
    "efflux",
]

fig, ax = plt.subplots(2, 3, figsize=(12, 8))
ax = ax.ravel()
for i, task in enumerate(tasks):
    biogen[task].hist(ax=ax[i])
    ax[i].set_title(task)
plt.tight_layout()
plt.show()

Typical of drug discovery assays, these tasks have a limited amount of data and are highly skewed. As a result, generalisation is a challenge and overfitting --- which may not be discernible if train and test data overlap --- is a likely outcome.

### Solubility task

We can also look specifically at certain subsets. Let's take a look at the solubility task data.

In [ ]:
solu = tp.data.load_dataset("Solu", verbose=True, root=temp_dir)
solu.head()

In task specfic datasets, the task column is relabelled to y:

In [ ]:
solu.y.hist()

And `rdkit_mols` maps to the molecules in that task dataset, or in this case subset.

In [ ]:
mols_to_plot = list(solu.rdkit_mols[:4]) + list(biogen.rdkit_mols[:4])
labels = [f"Solu {i}" for i in range(4)] + [f"Biogen {i}" for i in range(4)]
Chem.Draw.MolsToGridImage(mols_to_plot, molsPerRow=4, legends=labels)

### Clustering and splitting

For OOD train-test splits, we opted for Butina clustering on Morgan fingerprints and GroupKFold splitting.

In [ ]:
# setup a morgan fingerprints generator
# uses rdkit's implementation of morgan fingerprints
# asarray=False to get rdkit ExplicitBitVect as output instead of numpy arrays
morgan_generator = tp.data.mol.MorganGenerator(asarray=False)
morgan_generator

In [ ]:
fps: list[DataStructs.ExplicitBitVect] = morgan_generator(solu.rdkit_mols)

In [ ]:
# split the dataset using butina clustering on the morgan fingerprints
# uses groupkfolds with 5 repeats and 5 folds
# returns the splits and the cluster labels for each molecule
splits, clusters = tp.preprocess.butina_splitting(
    fps=fps,
    y=solu.y.values,
    threshold=0.65,
    repeats=5,
    kfolds=5,
    verbose=True,
    stratified=False,
)
f"Samples: {splits.shape[0]}, Splits: {splits.shape[1]}, Clusters: {clusters.shape[0]}"

In [ ]:
hist = plt.hist(clusters, bins=50)
plt.xlabel("Cluster ID")
plt.ylabel("Frequency")
plt.title("Cluster frenquencies")
plt.show()

Having split our data using repeated groupkfold and butina, we can train and test a model on the different splits.

### Training and testing a linear regression model on splits

In [ ]:
# get the morgan fingerprints as numpy arrays instead of rdkit ExplicitBitVect
morgan_generator.asarray = True
fps_array = morgan_generator(solu.rdkit_mols)

In [ ]:
# train a linear regression model on each split and calculate the r2 score on the test set
# Note: we're expecting low r2 scores since the relationship between morgan fingerprints and solubility is unlikely to be linear
# This is just to demonstrate how to use the splits
r2_scores = []
for spl in splits.T:
    train = spl == "Train"
    test = spl == "Test"
    X_train = fps_array[train]
    X_test = fps_array[test]
    y_train = solu.y.values[train]
    y_test = solu.y.values[test]
    model = LinearRegression(n_jobs=-1)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    r2_scores.append(r2)

In [ ]:
# plot the distribution of r2 scores across splits
plt.boxplot(r2_scores)
plt.xticks([1,], ["Linear Regression on Solubility Task"])
plt.title("R2 scores for linear regression on morgan fingerprints")
plt.show()

## QMugs

[QMugs](https://doi.org/10.1038/s41597-022-01390-7) was the primary dataset used for pre-training in this work.

Note: QMugs takes some time to download and generate standardized molecules for.

In [ ]:
# downloading QMugs dataset
standardizer=tp.data.mol.Standardizer(n_jobs=-1, verbose=True,)
qmugs = tp.data.load_dataset("QMugs", root=temp_dir, verbose=True, standardizer=standardizer,)

Investigating substructure data leakage in pre-training was a fundamental question in this work. To investigate this, we had to create filtered subsets of QMugs for pre-training based on Tanimoto similarity to the benchmark data. Below is an example of implementing this with just Solu (in the final study, all benchmark datasets were included).

In [ ]:
# get benchmark and pretrain fingerprints as list[ExplicitBitVect]
morgan_generator = tp.data.mol.MorganGenerator(
    fpsize=2048, radius=2,
    asarray=False, verbose=True
)
benchmark_fps = morgan_generator(solu.rdkit_mols)
rdkit_passes = qmugs[qmugs["rdkit_pass"]].index
rdkit_fails = qmugs[~qmugs["rdkit_pass"]].index

pretrain_fps = morgan_generator(qmugs.rdkit_mols[rdkit_passes])

In [ ]:
# calculate the maximum tanimoto similarity between each pretrain molecule and all benchmark molecules
max_tanimoto_scores_per_molecule = tp.preprocess.max_tanimoto(
    pretrain_fps, benchmark_fps, verbose=True
)
# get an array of 1s and 0s indicating which pretrain molecules 
# have a max tanimoto similarity below 0.5 to any benchmark molecule
pretrain_filter = tp.preprocess.float_to_binary(
    max_tanimoto_scores_per_molecule, threshold=0.5, below=True
)

In [ ]:
# add the pretrain filter to the qmugs dataframe
pretrain_filter_with_fails = np.empty(len(qmugs))
fails = 0
for (i, rd_pass) in enumerate(qmugs["rdkit_pass"]):
    if rd_pass:
        pretrain_filter_with_fails[i] = pretrain_filter[i - fails]
    else:
        pretrain_filter_with_fails[i] = 0
        fails += 1
qmugs["pretrain_filter"] = pretrain_filter_with_fails
qmugs.head()

In [ ]:
qmugs.pretrain_filter.value_counts()

## Running preprocessing

In [ ]:
# Command to run the preprocess script for preparing the benchmarking and pretraining datasets
# output_dir isn't used in the preprocess script, but we include it here for consistency with the main.py command format
# !python ../main.py --config config/preprocess/preprocess.yaml --data ../data --output ../output ../temp 

## Clear temp dir

In [ ]:
os.rmdir(temp_dir)